In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, concatenate, Average
from sklearn.model_selection import KFold # For training diversity
from sklearn.metrics import accuracy_score


2025-11-10 23:56:23.264922: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:


# --- 1. Define a function to create a Base Neural Network Model ---
def create_base_model(input_shape, hidden_units, output_units):
    inputs = Input(shape=input_shape)
    x = Dense(hidden_units, activation='relu')(inputs)
    x = Dense(hidden_units // 2, activation='relu')(x)
    outputs = Dense(output_units, activation='softmax')(x) # For classification
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model



In [4]:


# --- 2. Prepare Hypothetical Data ---
# Let's pretend we have data for a 10-class problem with 100 features
X_train = np.random.rand(1000, 100)
y_train = tf.keras.utils.to_categorical(np.random.randint(0, 10, 1000), num_classes=10)
X_test = np.random.rand(200, 100)
y_test = tf.keras.utils.to_categorical(np.random.randint(0, 10, 200), num_classes=10)

# --- 3. Create and Train Diverse Models ---
# Diversity is key! We can get it by:
# a) Different architectures (e.g., hidden_units)
# b) Different initial weights (different random seeds)
# c) Different training data subsets (Bagging - less common for full NNs due to cost)

input_shape = (X_train.shape[1],)
ensemble_members = []

# Model 1: Narrow Network
model_1 = create_base_model(input_shape, 64, 10)
model_1.fit(X_train, y_train, epochs=10, verbose=0)
ensemble_members.append(model_1)

# Model 2: Deeper Network
model_2 = create_base_model(input_shape, 128, 10)
model_2.fit(X_train, y_train, epochs=10, verbose=0)
ensemble_members.append(model_2)

# Model 3: Different Initialization/Training (just re-running the creation)
model_3 = create_base_model(input_shape, 64, 10)
model_3.fit(X_train, y_train, epochs=10, verbose=0)
ensemble_members.append(model_3)

# --- 4. The Ensemble: Soft Voting (Averaging Probabilities) ---
# Keras Functional API is essential for combining models in this way


# Get the predictions (probabilities) from each member
y_preds = [model.predict(X_test) for model in ensemble_members]

# Average the probabilities (Soft Voting)
# Summing the arrays and then dividing by the number of models (len(y_preds))
avg_preds = np.average(y_preds, axis=0) 

# Convert the averaged probabilities back to final class labels
ensemble_predictions = np.argmax(avg_preds, axis=1)
true_labels = np.argmax(y_test, axis=1)

# --- 5. Evaluate the Ensemble ---
ensemble_accuracy = accuracy_score(true_labels, ensemble_predictions)
print(f"Individual Model 1 Accuracy: {model_1.evaluate(X_test, y_test, verbose=0)[1]:.4f}")
print(f"Individual Model 2 Accuracy: {model_2.evaluate(X_test, y_test, verbose=0)[1]:.4f}")
print(f"Individual Model 3 Accuracy: {model_3.evaluate(X_test, y_test, verbose=0)[1]:.4f}")
print("-" * 30)
print(f"**Ensemble (Soft Voting) Accuracy: {ensemble_accuracy:.4f}**")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Individual Model 1 Accuracy: 0.0900
Individual Model 2 Accuracy: 0.1450
Individual Model 3 Accuracy: 0.0850
------------------------------
**Ensemble (Soft Voting) Accuracy: 0.1450**
